# 6.6 Automated Report Generation (Excel, HTML, PDF)

This notebook demonstrates how to automatically generate reports in different formats:
- **Excel** - Using `pandas` and `openpyxl`
- **HTML** - Using `pandas` and `Jinja2`
- **PDF** - Using `fpdf` library

## Install Required Libraries

In [1]:
# Install required packages
!pip install pandas openpyxl jinja2 fpdf xlsxwriter

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40770 sha256=4099f2ecb7494320d006feebf4e9b520bb4d24c0a80022db4e8eff601f039c84
  Stored in directory: c:\users\mausa\appdata\local\pip\cache\wheels\aa\da\11\a3189f34ddc13c26a2d0f329eac46b728c7f31c39e4dc26243
Successfully built fpdf

   ---------------------------------------- 0/2 [fpdf]
   ---------------------------------------- 0/2 [fpdf]
   -------------------- ------------------- 1/2 [xlsxwriter]
   -------------------- ------------------- 1/2 [xlsxwriter]
   -------------------- ------------------- 1/2 [xlsxwriter]
   -------------------- ------------------- 1/2 [xlsxwriter]
   ---------

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Create output directory for reports
os.makedirs('reports', exist_ok=True)
print("Report output directory created!")

Report output directory created!


## Create Sample Data

In [3]:
# Create sample sales data for demonstration
np.random.seed(42)

data = {
    'Product': ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard', 'Mouse', 'Headphones', 'Webcam'],
    'Category': ['Electronics', 'Electronics', 'Electronics', 'Electronics', 'Accessories', 'Accessories', 'Accessories', 'Accessories'],
    'Units_Sold': np.random.randint(50, 500, 8),
    'Unit_Price': [1200, 800, 450, 350, 75, 45, 150, 85],
    'Cost_Price': [900, 600, 350, 280, 50, 30, 100, 55]
}

df = pd.DataFrame(data)

# Calculate additional columns
df['Revenue'] = df['Units_Sold'] * df['Unit_Price']
df['Cost'] = df['Units_Sold'] * df['Cost_Price']
df['Profit'] = df['Revenue'] - df['Cost']
df['Profit_Margin'] = (df['Profit'] / df['Revenue'] * 100).round(2)

print("Sample Sales Data:")
df

Sample Sales Data:


,Product,Category,Units_Sold,Unit_Price,Cost_Price,Revenue,Cost,Profit,Profit_Margin
0,Laptop,Electronics,152,1200,900,182400,136800,45600,25.00
1,Phone,Electronics,485,800,600,388000,291000,97000,25.00
2,Tablet,Electronics,398,450,350,179100,139300,39800,22.22
3,Monitor,Electronics,320,350,280,112000,89600,22400,20.00
4,Keyboard,Accessories,156,75,50,11700,7800,3900,33.33
5,Mouse,Accessories,121,45,30,5445,3630,1815,33.33
6,Headphones,Accessories,238,150,100,35700,23800,11900,33.33
7,Webcam,Accessories,70,85,55,5950,3850,2100,35.29


---
# 1. Excel Report Generation

Using `pandas` with `openpyxl` or `xlsxwriter` engine for formatted Excel reports.

In [4]:
def generate_excel_report(df, filename='reports/sales_report.xlsx'):
    """
    Generate a formatted Excel report with multiple sheets.
    """
    # Create Excel writer with xlsxwriter engine for formatting
    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        workbook = writer.book
        
        # Define formats
        header_format = workbook.add_format({
            'bold': True,
            'bg_color': '#4472C4',
            'font_color': 'white',
            'border': 1,
            'align': 'center'
        })
        
        currency_format = workbook.add_format({'num_format': '$#,##0'})
        percent_format = workbook.add_format({'num_format': '0.00%'})
        
        # Sheet 1: Detailed Sales Data
        df.to_excel(writer, sheet_name='Sales Data', index=False, startrow=1)
        worksheet1 = writer.sheets['Sales Data']
        
        # Write header with formatting
        for col_num, value in enumerate(df.columns.values):
            worksheet1.write(0, col_num, value, header_format)
        
        # Auto-adjust column widths
        for i, col in enumerate(df.columns):
            max_length = max(df[col].astype(str).map(len).max(), len(col)) + 2
            worksheet1.set_column(i, i, max_length)
        
        # Sheet 2: Summary by Category
        summary = df.groupby('Category').agg({
            'Units_Sold': 'sum',
            'Revenue': 'sum',
            'Profit': 'sum'
        }).reset_index()
        summary['Avg_Profit_Margin'] = (summary['Profit'] / summary['Revenue'] * 100).round(2)
        
        summary.to_excel(writer, sheet_name='Category Summary', index=False, startrow=1)
        worksheet2 = writer.sheets['Category Summary']
        
        for col_num, value in enumerate(summary.columns.values):
            worksheet2.write(0, col_num, value, header_format)
        
        # Sheet 3: Dashboard/Overview
        overview_data = {
            'Metric': ['Total Revenue', 'Total Cost', 'Total Profit', 'Total Units Sold', 'Average Profit Margin'],
            'Value': [
                f"${df['Revenue'].sum():,}",
                f"${df['Cost'].sum():,}",
                f"${df['Profit'].sum():,}",
                f"{df['Units_Sold'].sum():,}",
                f"{df['Profit_Margin'].mean():.2f}%"
            ]
        }
        overview_df = pd.DataFrame(overview_data)
        overview_df.to_excel(writer, sheet_name='Dashboard', index=False, startrow=1)
        worksheet3 = writer.sheets['Dashboard']
        
        for col_num, value in enumerate(overview_df.columns.values):
            worksheet3.write(0, col_num, value, header_format)
        
        worksheet3.set_column(0, 0, 25)
        worksheet3.set_column(1, 1, 20)
    
    print(f"Excel report generated: {filename}")
    return filename

# Generate Excel Report
excel_file = generate_excel_report(df)
print("\nExcel report created successfully!")

Excel report generated: reports/sales_report.xlsx

Excel report created successfully!


---
# 2. HTML Report Generation

Using `pandas` built-in styling and `Jinja2` templates for professional HTML reports.

In [5]:
from jinja2 import Template

def generate_html_report(df, filename='reports/sales_report.html'):
    """
    Generate a styled HTML report using Jinja2 template.
    """
    # Calculate summary statistics
    total_revenue = df['Revenue'].sum()
    total_profit = df['Profit'].sum()
    total_units = df['Units_Sold'].sum()
    avg_margin = df['Profit_Margin'].mean()
    
    # Category summary
    category_summary = df.groupby('Category').agg({
        'Units_Sold': 'sum',
        'Revenue': 'sum',
        'Profit': 'sum'
    }).reset_index()
    
    # HTML Template
    html_template = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Sales Report - {{ report_date }}</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 40px;
                background-color: #f5f5f5;
            }
            .container {
                max-width: 1200px;
                margin: 0 auto;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            }
            h1 {
                color: #2c3e50;
                border-bottom: 3px solid #3498db;
                padding-bottom: 10px;
            }
            h2 {
                color: #34495e;
                margin-top: 30px;
            }
            .summary-cards {
                display: flex;
                gap: 20px;
                margin: 20px 0;
                flex-wrap: wrap;
            }
            .card {
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                padding: 20px;
                border-radius: 10px;
                flex: 1;
                min-width: 200px;
                text-align: center;
            }
            .card h3 {
                margin: 0;
                font-size: 14px;
                opacity: 0.9;
            }
            .card .value {
                font-size: 28px;
                font-weight: bold;
                margin-top: 10px;
            }
            table {
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }
            th, td {
                padding: 12px;
                text-align: left;
                border-bottom: 1px solid #ddd;
            }
            th {
                background-color: #3498db;
                color: white;
            }
            tr:hover {
                background-color: #f5f5f5;
            }
            .profit-positive {
                color: #27ae60;
                font-weight: bold;
            }
            .footer {
                margin-top: 40px;
                text-align: center;
                color: #7f8c8d;
                font-size: 12px;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>📊 Sales Performance Report</h1>
            <p>Report Generated: {{ report_date }}</p>
            
            <h2>Executive Summary</h2>
            <div class="summary-cards">
                <div class="card">
                    <h3>Total Revenue</h3>
                    <div class="value">${{ "{:,.0f}".format(total_revenue) }}</div>
                </div>
                <div class="card">
                    <h3>Total Profit</h3>
                    <div class="value">${{ "{:,.0f}".format(total_profit) }}</div>
                </div>
                <div class="card">
                    <h3>Units Sold</h3>
                    <div class="value">{{ "{:,}".format(total_units) }}</div>
                </div>
                <div class="card">
                    <h3>Avg Profit Margin</h3>
                    <div class="value">{{ "{:.1f}".format(avg_margin) }}%</div>
                </div>
            </div>
            
            <h2>Detailed Sales Data</h2>
            <table>
                <thead>
                    <tr>
                        {% for col in columns %}
                        <th>{{ col }}</th>
                        {% endfor %}
                    </tr>
                </thead>
                <tbody>
                    {% for row in data %}
                    <tr>
                        {% for item in row %}
                        <td>{{ item }}</td>
                        {% endfor %}
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
            
            <h2>Category Summary</h2>
            <table>
                <thead>
                    <tr>
                        {% for col in category_columns %}
                        <th>{{ col }}</th>
                        {% endfor %}
                    </tr>
                </thead>
                <tbody>
                    {% for row in category_data %}
                    <tr>
                        {% for item in row %}
                        <td>{{ item }}</td>
                        {% endfor %}
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
            
            <div class="footer">
                <p>This report was automatically generated using Python.</p>
            </div>
        </div>
    </body>
    </html>
    """
    
    # Render template
    template = Template(html_template)
    html_content = template.render(
        report_date=datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        total_revenue=total_revenue,
        total_profit=total_profit,
        total_units=total_units,
        avg_margin=avg_margin,
        columns=df.columns.tolist(),
        data=df.values.tolist(),
        category_columns=category_summary.columns.tolist(),
        category_data=category_summary.values.tolist()
    )
    
    # Save HTML file
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"HTML report generated: {filename}")
    return filename

# Generate HTML Report
html_file = generate_html_report(df)
print("\nHTML report created successfully!")

HTML report generated: reports/sales_report.html

HTML report created successfully!


### Simple HTML Report using Pandas Styler

In [6]:
def generate_simple_html_report(df, filename='reports/simple_report.html'):
    """
    Generate a simple HTML report using pandas built-in styling.
    """
    # Create styled dataframe
    styled = df.style\
        .background_gradient(subset=['Profit'], cmap='Greens')\
        .background_gradient(subset=['Revenue'], cmap='Blues')\
        .format({
            'Unit_Price': '${:,.0f}',
            'Cost_Price': '${:,.0f}',
            'Revenue': '${:,.0f}',
            'Cost': '${:,.0f}',
            'Profit': '${:,.0f}',
            'Profit_Margin': '{:.2f}%'
        })\
        .set_caption('Sales Performance Report')\
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'white')]},
            {'selector': 'caption', 'props': [('font-size', '20px'), ('font-weight', 'bold')]}
        ])
    
    # Save to HTML
    html_content = styled.to_html()
    with open(filename, 'w') as f:
        f.write(html_content)
    
    print(f"Simple HTML report generated: {filename}")
    return styled

# Generate Simple HTML Report
styled_df = generate_simple_html_report(df)
styled_df

Simple HTML report generated: reports/simple_report.html


,Product,Category,Units_Sold,Unit_Price,Cost_Price,Revenue,Cost,Profit,Profit_Margin
0,Laptop,Electronics,152,"$1,200",$900,"$182,400","$136,800","$45,600",25.00%
1,Phone,Electronics,485,$800,$600,"$388,000","$291,000","$97,000",25.00%
2,Tablet,Electronics,398,$450,$350,"$179,100","$139,300","$39,800",22.22%
3,Monitor,Electronics,320,$350,$280,"$112,000","$89,600","$22,400",20.00%
4,Keyboard,Accessories,156,$75,$50,"$11,700","$7,800","$3,900",33.33%
5,Mouse,Accessories,121,$45,$30,"$5,445","$3,630","$1,815",33.33%
6,Headphones,Accessories,238,$150,$100,"$35,700","$23,800","$11,900",33.33%
7,Webcam,Accessories,70,$85,$55,"$5,950","$3,850","$2,100",35.29%


---
# 3. PDF Report Generation

Using `fpdf` library to create professional PDF reports.

In [7]:
from fpdf import FPDF

class PDFReport(FPDF):
    """
    Custom PDF class for generating sales reports.
    """
    def header(self):
        self.set_font('Arial', 'B', 16)
        self.set_text_color(44, 62, 80)
        self.cell(0, 10, 'Sales Performance Report', 0, 1, 'C')
        self.set_font('Arial', '', 10)
        self.set_text_color(127, 140, 141)
        self.cell(0, 5, f'Generated on: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', 0, 1, 'C')
        self.ln(10)
    
    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(127, 140, 141)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')
    
    def chapter_title(self, title):
        self.set_font('Arial', 'B', 14)
        self.set_text_color(52, 73, 94)
        self.cell(0, 10, title, 0, 1, 'L')
        self.ln(2)
    
    def add_summary_card(self, label, value, x, y, width=45):
        self.set_xy(x, y)
        self.set_fill_color(52, 152, 219)
        self.set_text_color(255, 255, 255)
        self.set_font('Arial', 'B', 10)
        self.cell(width, 8, label, 0, 2, 'C', True)
        self.set_font('Arial', 'B', 12)
        self.cell(width, 10, str(value), 0, 1, 'C', True)
    
    def add_table(self, df, col_widths=None):
        # Header
        self.set_font('Arial', 'B', 9)
        self.set_fill_color(52, 152, 219)
        self.set_text_color(255, 255, 255)
        
        if col_widths is None:
            col_widths = [self.epw / len(df.columns)] * len(df.columns)
        
        for i, col in enumerate(df.columns):
            self.cell(col_widths[i], 8, str(col), 1, 0, 'C', True)
        self.ln()
        
        # Data rows
        self.set_font('Arial', '', 8)
        self.set_text_color(0, 0, 0)
        
        for index, row in df.iterrows():
            for i, item in enumerate(row):
                self.cell(col_widths[i], 7, str(item), 1, 0, 'C')
            self.ln()


def generate_pdf_report(df, filename='reports/sales_report.pdf'):
    """
    Generate a formatted PDF report.
    """
    pdf = PDFReport()
    pdf.add_page()
    
    # Executive Summary Section
    pdf.chapter_title('Executive Summary')
    
    # Summary Cards
    total_revenue = df['Revenue'].sum()
    total_profit = df['Profit'].sum()
    total_units = df['Units_Sold'].sum()
    avg_margin = df['Profit_Margin'].mean()
    
    start_y = pdf.get_y()
    pdf.add_summary_card('Total Revenue', f'${total_revenue:,.0f}', 10, start_y)
    pdf.add_summary_card('Total Profit', f'${total_profit:,.0f}', 60, start_y)
    pdf.add_summary_card('Units Sold', f'{total_units:,}', 110, start_y)
    pdf.add_summary_card('Avg Margin', f'{avg_margin:.1f}%', 160, start_y)
    
    pdf.ln(30)
    
    # Detailed Sales Table
    pdf.chapter_title('Detailed Sales Data')
    
    # Select key columns for PDF (to fit page width)
    pdf_df = df[['Product', 'Category', 'Units_Sold', 'Revenue', 'Profit', 'Profit_Margin']].copy()
    pdf_df['Revenue'] = pdf_df['Revenue'].apply(lambda x: f'${x:,.0f}')
    pdf_df['Profit'] = pdf_df['Profit'].apply(lambda x: f'${x:,.0f}')
    pdf_df['Profit_Margin'] = pdf_df['Profit_Margin'].apply(lambda x: f'{x:.1f}%')
    
    col_widths = [30, 30, 25, 35, 35, 30]
    pdf.add_table(pdf_df, col_widths)
    
    pdf.ln(10)
    
    # Category Summary
    pdf.chapter_title('Category Summary')
    
    category_summary = df.groupby('Category').agg({
        'Units_Sold': 'sum',
        'Revenue': 'sum',
        'Profit': 'sum'
    }).reset_index()
    category_summary['Revenue'] = category_summary['Revenue'].apply(lambda x: f'${x:,.0f}')
    category_summary['Profit'] = category_summary['Profit'].apply(lambda x: f'${x:,.0f}')
    
    cat_widths = [45, 40, 45, 45]
    pdf.add_table(category_summary, cat_widths)
    
    # Save PDF
    pdf.output(filename)
    print(f"PDF report generated: {filename}")
    return filename

# Generate PDF Report
pdf_file = generate_pdf_report(df)
print("\nPDF report created successfully!")

PDF report generated: reports/sales_report.pdf

PDF report created successfully!


---
# 4. Automated Report Generation Pipeline

Combining all report types into a single automated pipeline.

In [8]:
def automated_report_pipeline(df, report_name='sales', output_dir='reports'):
    """
    Automated pipeline to generate reports in all formats.
    
    Parameters:
    -----------
    df : pandas DataFrame
        Data to include in reports
    report_name : str
        Base name for report files
    output_dir : str
        Directory to save reports
    
    Returns:
    --------
    dict : Paths to generated reports
    """
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    reports = {}
    
    print("="*50)
    print("AUTOMATED REPORT GENERATION PIPELINE")
    print("="*50)
    
    # Generate Excel Report
    print("\n[1/3] Generating Excel Report...")
    excel_path = f"{output_dir}/{report_name}_{timestamp}.xlsx"
    generate_excel_report(df, excel_path)
    reports['excel'] = excel_path
    
    # Generate HTML Report
    print("\n[2/3] Generating HTML Report...")
    html_path = f"{output_dir}/{report_name}_{timestamp}.html"
    generate_html_report(df, html_path)
    reports['html'] = html_path
    
    # Generate PDF Report
    print("\n[3/3] Generating PDF Report...")
    pdf_path = f"{output_dir}/{report_name}_{timestamp}.pdf"
    generate_pdf_report(df, pdf_path)
    reports['pdf'] = pdf_path
    
    print("\n" + "="*50)
    print("REPORT GENERATION COMPLETE!")
    print("="*50)
    print("\nGenerated Reports:")
    for format_type, path in reports.items():
        print(f"  - {format_type.upper()}: {path}")
    
    return reports

# Run the automated pipeline
generated_reports = automated_report_pipeline(df, report_name='sales_performance')

AUTOMATED REPORT GENERATION PIPELINE

[1/3] Generating Excel Report...
Excel report generated: reports/sales_performance_20260213_155353.xlsx

[2/3] Generating HTML Report...
HTML report generated: reports/sales_performance_20260213_155353.html

[3/3] Generating PDF Report...
PDF report generated: reports/sales_performance_20260213_155353.pdf

REPORT GENERATION COMPLETE!

Generated Reports:
  - EXCEL: reports/sales_performance_20260213_155353.xlsx
  - HTML: reports/sales_performance_20260213_155353.html
  - PDF: reports/sales_performance_20260213_155353.pdf


---
# 5. Scheduling Report Generation (Bonus)

Example of how to schedule report generation using `schedule` library.

In [9]:
# Example of scheduling (not executed in this notebook)

"""
import schedule
import time

def daily_report_job():
    # Load fresh data
    df = pd.read_csv('sales_data.csv')
    
    # Generate reports
    automated_report_pipeline(df, report_name='daily_sales')
    
    print(f"Daily reports generated at {datetime.now()}")

# Schedule daily report at 9:00 AM
schedule.every().day.at("09:00").do(daily_report_job)

# Run scheduler
while True:
    schedule.run_pending()
    time.sleep(60)
"""

print("Scheduling example code provided above (commented out).")
print("Use 'schedule' library or system CRON jobs for automated scheduling.")

Scheduling example code provided above (commented out).
Use 'schedule' library or system CRON jobs for automated scheduling.


---
## Summary

In this notebook, we learned:

1. **Excel Reports** - Using `pandas` with `xlsxwriter` for formatted multi-sheet Excel reports
2. **HTML Reports** - Using `Jinja2` templates for professional styled web reports
3. **PDF Reports** - Using `fpdf` library for printable PDF documents
4. **Automated Pipeline** - Combining all formats into a single automated workflow

### Key Libraries Used:
- `pandas` - Data manipulation and Excel export
- `openpyxl` / `xlsxwriter` - Excel formatting
- `jinja2` - HTML templating
- `fpdf` - PDF generation

In [10]:
# List all generated reports
print("Generated Reports in 'reports' folder:")
for file in os.listdir('reports'):
    print(f"  - {file}")

Generated Reports in 'reports' folder:
  - sales_performance_20260213_155353.html
  - sales_performance_20260213_155353.pdf
  - sales_performance_20260213_155353.xlsx
  - sales_report.html
  - sales_report.pdf
  - sales_report.xlsx
  - simple_report.html
